# BaseAgent - Parkinson

This notebook contains chronical records of BaseAgent's attempt in recreating a Parkinson's Disease knowledge graph.

In [3]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import os
import shutil

from BaseAgent import BaseAgent, AgentTeam, MaxRoundsExceededError
from BaseAgent.agent_spec import AgentSpec


MCP_CONFIG = "examples/mcp_config.yaml"
SKILLS_DIR = "skills"
TEMPLATE_SRC = os.path.abspath("template")
PROJECT_NAME = "parkinsondb"
DISPLAY_NAME = "Parkinson's Knowledge Base"
THREAD_ID = PROJECT_NAME
# copy the template to a new project folder to work with
shutil.copytree("template", PROJECT_NAME)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


'parkinsondb'

In [4]:
def _print_token_summary(agents: list):
    """Print per-agent and total token counts from accumulated usage_metrics."""
    print("\n=== Token usage ===")
    totals = {"input": 0, "output": 0, "total": 0}
    for agent in agents:
        metrics = agent.usage_metrics
        input_tokens = sum(m.input_tokens or 0 for m in metrics)
        cache_creation_tokens = sum(m.cache_creation_tokens or 0 for m in metrics)
        cache_read_tokens = sum(m.cache_read_tokens or 0 for m in metrics)
        output_tokens = sum(m.output_tokens or 0 for m in metrics)
        total_tokens = sum(m.total_tokens or 0 for m in metrics)
        cost = sum(m.cost or 0.0 for m in metrics)
        cost_str = f"  ${cost:.4f}" if cost else ""
        print(f"  {agent.spec.name}: {input_tokens} in / {output_tokens} out / {cache_creation_tokens} cache creation / {cache_read_tokens} cache read / {total_tokens} total{cost_str}")
        totals["input"] += input_tokens
        totals["output"] += output_tokens
        totals["cache_creation"] += cache_creation_tokens
        totals["cache_read"] += cache_read_tokens
        totals["total"] += total_tokens
    print(f"  {'─' * 40}")
    print(f"  all agents:  {totals['input']} in / {totals['output']} out / {totals['cache_creation']} cache creation / {totals['cache_read']} cache read / {totals['total']} total")

# Run a KG-building agentic AI team
Re-run this block when you need to refresh the agents, for example, clear up agent memory or context.

In [ ]:
# ----- Ontology Agent -----
ontology_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="ontology_agent",
        role=(
            f"A biomedical ontology engineer managing the OWL schema and project configuration. "
            f"You own {PROJECT_NAME}/data/ontology/ontology.rdf and {PROJECT_NAME}/config/project.yaml: update disease_scope "
            f"(primary_terms, umls_cuis, doid_ids, mesh_ids) and keep node_types/edge_types in "
            f"sync with the RDF. Only modify the RDF on explicit request. Never edit Python source files."
            "Never read the full RDF into memory - use rdflib to query only the relevant sections for each task."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["ontology-protocol"],
    ),
    require_approval="never",
)
ontology_agent.add_mcp(MCP_CONFIG)

# ----- Database Agent -----
database_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="database_agent",
        role=(
            f"A bioinformatics data engineer managing {PROJECT_NAME}/config/databases.yaml. "
            f"You evaluate biomedical data sources and set enabled flags for the target disease. "
            f"Never edit parsers or ontology mappings."
            "Never read the full RDF, TSV, or database dumps into memory - use streaming or chunking to process only the relevant sections for each task."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["database-protocol"],
    ),
    require_approval="never",
)
database_agent.add_mcp(MCP_CONFIG)

# ----- Engineer Agent -----
engineer_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="engineer_agent",
        role=(
            f"A Python software engineer writing parsers under {PROJECT_NAME}/src/parsers/. "
            f"Each parser inherits from BaseParser and downloads data from one biomedical source, "
            f"returning clean pandas DataFrames. Follow the full registration checklist: "
            f"{PROJECT_NAME}/src/parsers/__init__.py and {PROJECT_NAME}/src/main.py PARSERS dict. "
            f"Run `python {PROJECT_NAME}/src/main.py --source <name>` to verify each parser produces TSVs. "
            f"Never hardcode any disease-specific values in the parser code. "
            f"Never hardcode any credentials."
            f"Never modify OWL files or ontology_mappings.yaml."
            "Never read the full RDF, TSV, or database dumps into memory - use streaming or chunking to process only the relevant sections for each task."
        ),
        llm="azure-claude-sonnet-4-6",
        skill_names=["parser-protocol"],
    ),
    require_approval="never",
)
engineer_agent.add_mcp(MCP_CONFIG)

# ----- Mapping Agent -----
mapping_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="mapping_agent",
        role=(
            f"A knowledge graph mapping specialist owning {PROJECT_NAME}/config/ontology_mappings.yaml. "
            f"You map processed TSV columns to OWL node types and relationship types. "
            f"Other useful information such as gene descriptions, annotations, structures, and cross-references is included as properties. "
            f"Always place node entries before relationship entries. "
            f"Verify all OWL names against {PROJECT_NAME}/config/project.yaml node_types/edge_types before writing. "
            f"Never edit Python source files."
            "Never read the full RDF, TSV, or database dumps into memory - use streaming or chunking to process only the relevant sections for each task."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["mapping-protocol"],
    ),
    require_approval="never",
)
mapping_agent.add_mcp(MCP_CONFIG)

# ----- Memgraph Agent -----
memgraph_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="memgraph_agent",
        role=(
            f"A graph database engineer who runs the full pipeline and validates graph export. "
            f"Run `python {PROJECT_NAME}/src/main.py` inside the repo to produce {PROJECT_NAME}/data/output/ontology_populated.rdf."
            "Never read the full RDF into memory - use rdflib to query only the relevant sections for each task. "
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["memgraph-protocol"],
    ),
    require_approval="never",
)
memgraph_agent.add_mcp(MCP_CONFIG)

# ----- Evaluator Agent -----
evaluator_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="evaluator_agent",
        role=(
            f"A KG quality evaluator who runs {PROJECT_NAME}/eval/eval_after_parser.py, {PROJECT_NAME}/eval/eval_after_ontology.py, "
            f"and {PROJECT_NAME}/eval/eval_after_memgraph.py in sequence. Report tier-1 blocking failures "
            f"(zero node/edge counts, OWL conformance < 1.0) and overall KG quality. "
            f"Flag any blocking failures that must be resolved before the KG is used."
            "Never read the full RDF, TSV, or database dumps into memory - use streaming or chunking to process only the relevant sections for each task. "
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["evaluation-protocol"],
    ),
    require_approval="never",
)
evaluator_agent.add_mcp(MCP_CONFIG)

# ----- HITL Agent -----
hitl_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="hitl_agent",
        role=(
            "A human review coordinator. "
            "Call `ask_user` with a drafted message that summarizes the previous agent's output in less than 5 bullet."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["hitl-protocol"],
    ),
    require_approval="never",
)
hitl_agent.add_mcp(MCP_CONFIG)

# ----- Graph Exporter Agent -----
graph_exporter_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="graph_exporter_agent",
        role=(
            f"A graph database engineer who runs the full pipeline and validates graph export. "
            f"Run `python {PROJECT_NAME}/src/main.py` inside the repo to produce {PROJECT_NAME}/data/output/ontology_populated.rdf, "
            f"then run MemgraphExporter. Validate import.cypher (INDEX statements, LOAD CSV paths, "
            f"globally unique node IDs). Provide docker run import instructions."
            "Never read the full RDF, TSV, or database dumps into memory - use streaming or chunking to process only the relevant sections for each task. "
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["memgraph-protocol"],
    ),
    require_approval="never",
)
graph_exporter_agent.add_mcp(MCP_CONFIG)

Auto-detecting source from model name: azure-claude-haiku-4-5
Creating AnthropicFoundry model: azure-claude-haiku-4-5
Skill 'ontology-protocol' successfully added

🔧 BASE AGENT CONFIGURATION
Agent Name: ontology_agent
LLM: azure-claude-haiku-4-5
Source: AnthropicFoundry
Base URL: None
Loaded Tools: ['run_python_repl', 'read_function_source_code']
Use Tool Retriever: False

Discovered 14 tools from filesystem MCP server (local)
Auto-detecting source from model name: azure-claude-haiku-4-5
Creating AnthropicFoundry model: azure-claude-haiku-4-5
Skill 'database-protocol' successfully added

🔧 BASE AGENT CONFIGURATION
Agent Name: database_agent
LLM: azure-claude-haiku-4-5
Source: AnthropicFoundry
Base URL: None
Loaded Tools: ['run_python_repl', 'read_function_source_code']
Use Tool Retriever: False

Discovered 14 tools from filesystem MCP server (local)
Auto-detecting source from model name: azure-claude-sonnet-4-6
Creating AnthropicFoundry model: azure-claude-sonnet-4-6
Skill 'parser-prot

# Build the whole knowledge graph

In [ ]:

agents = [
    ontology_agent, database_agent, engineer_agent, evaluator_agent, mapping_agent, hitl_agent, graph_exporter_agent
]

team = AgentTeam(
    agents=agents,
    supervisor_llm="azure-claude-opus-4-6",
    max_rounds=50,
)

task = f"""
Coordinate a team of agents to build a Parkinson's disease knowledge graph using {PROJECT_NAME}.

The ONLY configuration change needed before running the pipeline is updating disease scope in config/project.yaml:
  name: {PROJECT_NAME}
  display_name: "{DISPLAY_NAME}"
  disease_scope:
    primary_terms: ["parkinson"]
    umls_cuis: ["C0030567"]
    doid_ids: ["DOID:14330"]
    mesh_ids: ["D010300"]

Strict contracts:
- After ontology_agent updates config/project.yaml, run the full pipeline without any other changes.
- If there is any parser failure, report the failure and skip the source.

Available agents and their responsibilities:
- ontology_agent: Update disease scope in config/project.yaml only. No RDF changes needed.
- database_agent: Manage config/databases.yaml — enable or add data sources.
- engineer_agent: Run `python src/main.py --source <name>` inside the repo to verify each parser.
- mapping_agent: Manage config/ontology_mappings.yaml and populate ontology — ensure the correct configurations (RDF classes, data property map, and merge/cross-reference).
- evaluator_agent: Run eval_after_parser.py and eval_after_ontology.py. Report tier-1 blocking failures.
- graph_exporter_agent: Run the full pipeline with `python src/main.py` to produce data/output/ontology_populated.rdf. Validate import.cypher and provide docker run import instructions.
- hitl_agent: Pause for user review before any config changes. Relay feedback to the responsible agent.

Inter-agent contracts:
- Evaluator_agent must run before graph_exporter_agent.
- On tier-1 failures, route back to engineer_agent to fix before re-evaluating.
"""

Auto-detecting source from model name: azure-claude-opus-4-6
Creating AnthropicFoundry model: azure-claude-opus-4-6


In [7]:
_log, result = team.run_sync(f"{task}", thread_id=THREAD_ID)


[supervisor] → hitl_agent: Before we begin building the Parkinson's disease knowledge graph, pause for user review of the planned configuration change. Summarize the following for the user and ask for approval:

- We will update config/project.yaml to set the disease scope for Parkinson's disease.
- primary_terms will be set to ["parkinson"].
- umls_cuis will be set to ["C0030567"].
- doid_ids will be set to ["DOID:14330"].
- mesh_ids will be set to ["D010300"].
- Project name will be "parkinsondb" with display_name "Parkinson's Knowledge Base".

Ask the user to confirm before proceeding with the config change.

[hitl_agent] Starting
================================ Human Message =================================

Before we begin building the Parkinson's disease knowledge graph, pause for user review of the planned configuration change. Summarize the following for the user and ask for approval:

- We will update config/project.yaml to set the disease scope for Parkinson's disease.
- p

In [ ]:
_log, result = team.run_sync(f"""
    {task}. 
    Previous task was interrupted due to Anthropic server overload. 
    Find out the latest progress and continue with the task.""", 
    thread_id=THREAD_ID
    )

In [ ]:
_print_token_summary(agents)

# Building individual database parser

In [ ]:

agents = [
    ontology_agent, database_agent, engineer_agent, evaluator_agent, mapping_agent, hitl_agent,
]

team = AgentTeam(
    agents=[ontology_agent, database_agent, engineer_agent, 
            evaluator_agent, mapping_agent, hitl_agent, graph_exporter_agent],
    supervisor_llm="azure-claude-opus-4-6",
    max_rounds=50,
)


TARGET_DATABASE = "ncbi gene"
THREAD_ID = TARGET_DATABASE

task = f"""
Coordinate a team of agents to update and evaluate the {TARGET_DATABASE} parser in {TEMPLATE_SRC}. All files are in {TEMPLATE_SRC}. 

Available agents and their responsibilities:
- ontology_agent: Manage OWL schema in data/ontology/ontology.rdf and config/project.yaml — ensure the project strictly conforms to the ontology.
- database_agent: Manage config/databases.yaml — enable or add data sources.
- engineer_agent: Manage parsers in src/parsers/. Verify with `python src/main.py --source <name>` run inside the repo.
- mapping_agent: Manage config/ontology_mappings.yaml — ensure the correct configurations (RDF classes, data property map, and merge/cross-reference).
- evaluator_agent: Focus on parser-specifc evaluation by running only eval_after_parser script and report the quality summary. List any tier-1 blocking failures explicitly.
- hitl_agent: Pause for user review when proposing changes to config files (config/*.yaml). Relay any feedback back to the responsible agent.


Instructions:
- Find more information about the {TARGET_DATABASE} data and the download file contents from the official {TARGET_DATABASE} documentation.
- Include useful information as properties. For drugs, useful information includes the drug description, index, structure (in SMILES and InChI formats), FDA approval status, and more. For diseases, useful information include disease descriptions, symptoms, and other relevant details. For genes, useful information includes expression, call quality, FDR, expression score, and expression rank.
- Coordinate across agents to produce a verified {TARGET_DATABASE} parser with no tier-1 blocking failures.
- Add new RDF classes when and only when the extracted data can be cross referenced to other sources using standardized identifiers. Free-text descriptions can only be included as properties, not RDF classes.
- Upon user approval, update OWL schema in data/ontology/ontology.rdf and graph schema in config/ontology_mappings.yaml to accommodate the new RDF classes.
- Upon user approval, update "data_property_map" in config/ontology_mappings.yaml to accommodate the new RDF properties.


Inter-agent contracts:
- All parsers must be verified before evaluator_agent runs.
- mapping_agent must report valid cross-reference information before the engineer_agent starts.
- On tier-1 failures, route back to engineer_agent to fix before re-evaluating.


Strict contracts and constraints:
- Do not include any filter (e.g., disease, tissue, or drug) if the specific parser doesn't include these RDF classes.
- Directly retrieve data from the source database. Do not use any precomputed third-party processed data. Do not use any precomputed data from Hetionet.
- config/project.yaml sets project.name, display_name, and all disease_scope fields (primary_terms, umls_cuis, doid_ids, mesh_ids). 
- The databases.yaml key, PARSERS key, PARSER_CLASS_MAP key, ontology_mappings.yaml prefix, and data/processed/ subdirectory name must all be identical strings.
- In ontology_mappings.yaml, all node entries must precede all relationship entries.
- In ontology_mappings.yaml, merge must be true.
- OWL names in ontology_mappings.yaml must be active (uncommented) in project.yaml.
- Credentials use the _env suffix in databases.yaml args; the parser constructor must accept the stripped parameter name.
- Do minimal changes.

{TARGET_DATABASE} specific instructions:
- Extract the node type: Gene.
"""

In [ ]:
_log, result = team.run_sync(f"{task}", thread_id=THREAD_ID)

In [ ]:
_log, result = team.run_sync(f"summarize the current progress and issues.", thread_id=THREAD_ID)

In [ ]:
_print_token_summary(agents)

# Sub-agent follow-up

In [9]:
_log2, result2 = graph_exporter_agent.run(
    f"ista is now correctly installed. Re-run the export step.",
    thread_id=f"{THREAD_ID}:graph_exporter_agent"
)

================================ Human Message =================================

ista is now correctly installed. Re-run the export step.
================================== Ai Message ==================================

<think>
The user wants me to re-run the export step for the graph database. This means I need to:

1. First, verify that the populated RDF file exists at `parkinsondb/data/output/ontology_populated.rdf`
2. Run the MemgraphExporter to generate the CSV and Cypher files
3. Validate the outputs (import.cypher, CSV files, etc.)
4. Provide Docker run instructions

Let me start by making a plan and then executing it step by step.
</think>

<execute>
# Let me create a detailed plan for re-running the export step

plan = """
PLAN: Re-run Graph Export and Validation

1. [ ] Verify populated RDF exists at parkinsondb/data/output/ontology_populated.rdf
2. [ ] Run MemgraphExporter to generate CSV and Cypher files
3. [ ] Verify export outputs:
   - nodes_*.csv files exist
   - edges